# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [8]:
!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

In [1]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [2]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [3]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - and download our webpages which we'll be using for our data today.

These webpages are from [Simon Willison's](https://simonwillison.net/) yearly "AI learnings".

- [2023 Blog](https://simonwillison.net/2023/Dec/31/ai-in-2023/)
- [2024 Blog](https://simonwillison.net/2024/Dec/31/llms-in-2024/)

Let's start by collecting our data into a useful pile!

In [4]:
!mkdir data

mkdir: data: File exists


In [5]:
!curl https://simonwillison.net/2023/Dec/31/ai-in-2023/ -o data/2023_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 31440    0 31440    0     0  66800      0 --:--:-- --:--:-- --:--:--     0:--:-- --:--:-- --:--:-- 66751


In [6]:
!curl https://simonwillison.net/2024/Dec/31/llms-in-2024/ -o data/2024_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 70299    0 70299    0     0   738k      0 --:--:-- --:--:-- --:--:--  746k


Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [3]:
from langchain_community.document_loaders import DirectoryLoader

path = "data/"
loader = DirectoryLoader(path, glob="*.html")
docs = loader.load()

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [18]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/opt/homebrew/Caskroom/miniforge/base/envs/llmops-course/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [19]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [20]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [21]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

KnowledgeGraph(nodes: 14, relationships: 68)

We can save and load our knowledge graphs as follows.

In [22]:
kg.save("ai_across_years_kg.json")
ai_across_years_kg = KnowledgeGraph.load("ai_across_years_kg.json")
ai_across_years_kg

KnowledgeGraph(nodes: 14, relationships: 68)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [23]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=ai_across_years_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [24]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

LSY ANSWER ->
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
            creates simple questions that need just one piece of information 
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
            creates questions that need you to connect data between different parts of a document
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
            creates questions that need multiple specific facts, like asking to combine details from different sections of a document

Finally, we can use our `TestSetGenerator` to generate our testset!

In [25]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating Samples: 100%|██████████| 11/11 [00:12<00:00,  1.10s/it]


,user_input,reference_contexts,reference,synthesizer_name
0,What significant developments in large languag...,[Code may be the best application The ethics o...,"In 2024, it was observed that large language m...",single_hop_specifc_query_synthesizer
1,Wht iz the role of JavaScript in the context o...,[Based Development As a computer scientist and...,"JavaScript, as a programming language, is ment...",single_hop_specifc_query_synthesizer
2,"What stuff we figured out about AI in 2023, li...",[Simon Willison’s Weblog Subscribe Stuff we fi...,"In 2023, it was a breakthrough year for Large ...",single_hop_specifc_query_synthesizer
3,What are the ethical implications of using DAL...,[easy to follow. The rest of the document incl...,The ethical implications of using DALL-E 3 for...,single_hop_specifc_query_synthesizer
4,What advancements has Anthropic made in large ...,[Prompt driven app generation is a commodity a...,Anthropic launched the Claude 3 series in Marc...,single_hop_specifc_query_synthesizer
5,How has the concept of universal access to AI ...,[<1-hop>\n\nPrompt driven app generation is a ...,The concept of universal access to AI models e...,multi_hop_abstract_query_synthesizer
6,How do the challenges of gullibility in AI per...,[<1-hop>\n\nPrompt driven app generation is a ...,The challenges of gullibility in AI personal a...,multi_hop_abstract_query_synthesizer
7,How has the concept of universal access to AI ...,[<1-hop>\n\nPrompt driven app generation is a ...,"In 2024, the concept of universal access to AI...",multi_hop_abstract_query_synthesizer
8,How have advancements in large language models...,[<1-hop>\n\nPrompt driven app generation is a ...,Advancements in large language models like GPT...,multi_hop_specific_query_synthesizer
9,What are the cost and efficiency benefits of u...,[<1-hop>\n\neasy to follow. The rest of the do...,Google's Gemini 1.5 Flash model offers signifi...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [26]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Generating Samples: 100%|██████████| 12/12 [01:03<00:00,  5.32s/it]


In [29]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What Mistral do?,[Code may be the best application The ethics o...,Mistral is one of the organizations that have ...,single_hop_specifc_query_synthesizer
1,Wht iz the role of JavaScript in the context o...,[Based Development As a computer scientist and...,"JavaScript, as a programming language, is one ...",single_hop_specifc_query_synthesizer
2,Wht are the key insights from Simon Willison's...,[Simon Willison’s Weblog Subscribe Stuff we fi...,Simon Willison's Weblog highlights that 2023 w...,single_hop_specifc_query_synthesizer
3,Wht are the Plausible analytics insights from ...,[easy to follow. The rest of the document incl...,The AI Research Analyst used Plausible analyti...,single_hop_specifc_query_synthesizer
4,How have inference-scaling reasoning models an...,[<1-hop>\n\nPrompt driven app generation is a ...,"In 2024, the advancements in large language mo...",multi_hop_abstract_query_synthesizer
5,How have inference-scaling reasoning models an...,[<1-hop>\n\nPrompt driven app generation is a ...,Inference-scaling reasoning models have been a...,multi_hop_abstract_query_synthesizer
6,How have inference-scaling reasoning models an...,[<1-hop>\n\nPrompt driven app generation is a ...,Inference-scaling reasoning models have been a...,multi_hop_abstract_query_synthesizer
7,How have advancements in large language models...,[<1-hop>\n\nPrompt driven app generation is a ...,"Advancements in large language models, such as...",multi_hop_abstract_query_synthesizer
8,What happen with large language models in 2023...,[<1-hop>\n\nSimon Willison’s Weblog Subscribe ...,"In 2023, large language models (LLMs) were rec...",multi_hop_specific_query_synthesizer
9,How has Google's Gemini 1.5 Flash model contri...,[<1-hop>\n\ngets you OpenAI’s most expensive m...,Google's Gemini 1.5 Flash model has significan...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [37]:
from langsmith import Client
from datetime import datetime

client = Client()

dataset_name = "State of AI Across the Years"

# Try to delete the existing dataset if it exists
try:
    existing_dataset = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_name=dataset_name)
except:
    pass  # Dataset doesn't exist, which is fine

# Create new dataset
langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="State of AI Across the Years!"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [38]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [39]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [40]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [41]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [42]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="State of AI"
)

In [43]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [44]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [45]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

Finally, we can set-up our RAG LCEL chain!

In [46]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [47]:
rag_chain.invoke({"question" : "What are Agents?"})

'Agents are described as AI systems that can act on your behalf, but the term is considered vague and lacks a clear, widely understood definition. There are two main categories of understanding regarding agents: one sees them as systems that act like a travel agent, while the other views them as LLMs (large language models) that utilize tools to solve problems. The concept of autonomy is also mentioned, but again without a clear definition. Overall, there is skepticism about their utility due to issues like gullibility in AI systems.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4o as our evaluation LLM for our base Evaluators.

In [48]:
eval_llm = ChatOpenAI(model="gpt-4o")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [49]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dope_or_nope_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this submission dope, lit, or cool?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

LSY ANSWER -> 
- `qa_evaluator`: Evaluates the accuracy of the answer by comparing it to the reference answer.
- `labeled_helpfulness_evaluator`: Evaluates the helpfulness of the answer by considering the relevance to the user's question and the correctness of the reference answer.
- `dope_or_nope_evaluator`: Evaluates the coolness of the answer by checking if it's dope, lit, or cool.

## LangSmith Evaluation

In [50]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dope_or_nope_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'back-meat-58' at:
https://smith.langchain.com/o/fb0d5d7c-8599-4482-8163-c1156c9773c9/datasets/6a46751c-70f8-4da7-8020-b78327aba3d4/compare?selectedSessions=196902d8-d505-4cb1-946f-592822af61e2




12it [02:24, 12.00s/it]


,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How has the development of Claude 3 and its su...,I don't know.,None,The development of Claude 3 and its subsequent...,0,0,0,3.667224,9e452c39-e383-40df-8939-0ad43669c95a,875be98b-4811-4b1c-9c91-e96914a01f24
1,What advancements in large language models wer...,"In 2024, significant advancements in large lan...",None,"In 2024, significant advancements in large lan...",1,0,1,2.734534,62325937-d2fa-4fe3-b0fc-639ca8476bff,91cde943-8c53-479b-b10f-84bc6b46f5bc
2,How has Google's Gemini 1.5 Flash model contri...,I don't know.,None,Google's Gemini 1.5 Flash model has significan...,0,0,0,1.041966,12d472f9-8fbd-4032-9949-0da0c10523ec,484d67be-9dab-4421-b454-b316e6bc3507
3,What happen with large language models in 2023...,"In 2023, Large Language Models (LLMs) experien...",None,"In 2023, large language models (LLMs) were rec...",1,1,1,3.774387,876fe1a2-97de-43ca-8c25-cdb669d15fbb,ff89eb4b-7305-4b63-bff3-1d6b60e43589
4,How have advancements in large language models...,"Advancements in large language models, such as...",None,"Advancements in large language models, such as...",1,0,0,3.927562,cfb7fba3-3f47-4bca-983b-fc8ba07343ca,0817e456-05e9-4d4a-9da3-0932eae5ee92
5,How have inference-scaling reasoning models an...,"Inference-scaling reasoning models, such as Op...",None,Inference-scaling reasoning models have been a...,1,0,0,2.838548,71a343ef-936b-4ad6-addf-be5bdae84fe4,287d6a61-526c-4d09-98f7-873d401f88cb
6,How have inference-scaling reasoning models an...,"Inference-scaling reasoning models, exemplifie...",None,Inference-scaling reasoning models have been a...,1,0,1,7.529293,541fd9c7-9998-42bc-bd4a-fb4f5e96059a,662c347b-2071-4518-8d50-c0dd8ae3a7ba
7,How have inference-scaling reasoning models an...,I don't know.,None,"In 2024, the advancements in large language mo...",0,0,0,0.869657,9cecc220-6ddf-45d8-8af9-ef3e106ba88e,a4322375-d37f-4edc-9787-be3c537ee1d4
8,Wht are the Plausible analytics insights from ...,I don't know.,None,The AI Research Analyst used Plausible analyti...,0,0,0,0.875736,ad4d804f-9984-4feb-b4cd-7542f6cb7380,bada1688-e1d2-490d-9ad4-09ba22064acc
9,Wht are the key insights from Simon Willison's...,Key insights from Simon Willison's Weblog abou...,None,Simon Willison's Weblog highlights that 2023 w...,1,0,0,2.595741,b148588d-9c6d-4726-9136-5c26eef82d50,38786e8d-674b-4a52-96e1-254a18d8f1c3


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [2]:
DOPE_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the questions in a dope way, be cool!

Context: {context}
Question: {question}
"""

dope_rag_prompt = ChatPromptTemplate.from_template(DOPE_RAG_PROMPT)

NameError: name 'ChatPromptTemplate' is not defined

In [52]:
rag_documents = docs

In [53]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

LSY ANSWER -> 
Retrieval Efficiency – Smaller chunks improve granularity but increase retrieval latency; larger chunks reduce retrieval calls but may dilute relevance.
Embedding & Indexing Cost – Smaller chunks create more embeddings, increasing storage and processing overhead.
Response Quality – Smaller chunks risk losing context; larger chunks retain context but may retrieve irrelevant content.
Overlap & Redundancy – Higher chunk_overlap improves continuity but increases redundancy.
Latency & Performance – Smaller chunks mean more retrievals, increasing processing time; larger chunks improve speed but risk lower relevance.

In [54]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?
LSY ANSWER -> 
Embedding Quality – Larger models (e.g., text-embedding-3-large) capture richer semantics, improving retrieval accuracy, while smaller models may miss nuances.
Indexing & Storage Cost – Higher-dimensional embeddings require more storage and increase indexing time.
Retrieval Latency – Larger embeddings slow down vector search and increase compute cost, while smaller embeddings speed up retrieval.
Model Cost – More advanced models incur higher API costs per call.
Downstream Task Performance – Better embeddings improve response relevance but may introduce diminishing returns if retrieval isn't the bottleneck.

In [55]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="AI Across Years (Augmented)"
)

In [56]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [57]:
dope_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dope_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [58]:
dope_rag_chain.invoke({"question" : "what are Agents?"})

'Agents are like those elusive creatures in the AI realm that are supposed to act on your behalf, but the term is super vague and lacks a clear definition. Some folks think of them as digital assistants or travel agents, while others see them as LLMs running tools to solve problems. But honestly, it feels like they’ve been “coming soon” for ages, with not much solid action in the real world yet. So, yeah, the whole agent scene is still a bit of a mystery!'

Finally, we can evaluate the new chain on the same test set!

In [59]:
evaluate(
    dope_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dope_or_nope_evaluator
    ],
    metadata={"revision_id": "dope_chain"},
)

View the evaluation results for experiment: 'sunny-bag-54' at:
https://smith.langchain.com/o/fb0d5d7c-8599-4482-8163-c1156c9773c9/datasets/6a46751c-70f8-4da7-8020-b78327aba3d4/compare?selectedSessions=1c272251-a23e-40da-944a-47047c0ed248




12it [02:26, 12.18s/it]


,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How has the development of Claude 3 and its su...,"Yo, the evolution of Claude 3 and its offsprin...",None,The development of Claude 3 and its subsequent...,1,1,1,3.880808,9e452c39-e383-40df-8939-0ad43669c95a,1a448e6e-1e57-4a7a-bf14-49f9294754f8
1,What advancements in large language models wer...,"Oh, you wanna know about the cool stuff that w...",None,"In 2024, significant advancements in large lan...",1,0,1,3.590829,62325937-d2fa-4fe3-b0fc-639ca8476bff,6dd19d20-912a-4e0f-b3c2-e1b46e728791
2,How has Google's Gemini 1.5 Flash model contri...,"Yo, the Gemini 1.5 Flash model from Google is ...",None,Google's Gemini 1.5 Flash model has significan...,1,0,1,2.930267,12d472f9-8fbd-4032-9949-0da0c10523ec,4f0efa7d-6dee-4554-9358-9051ae1779aa
3,What happen with large language models in 2023...,"Yo, in 2023, Large Language Models (LLMs) hit ...",None,"In 2023, large language models (LLMs) were rec...",1,1,1,2.302991,876fe1a2-97de-43ca-8c25-cdb669d15fbb,fedb671f-befc-4e7a-9eac-cb9496933624
4,How have advancements in large language models...,"Yo, check it out! The advancements in large la...",None,"Advancements in large language models, such as...",1,0,1,3.616383,cfb7fba3-3f47-4bca-983b-fc8ba07343ca,80336584-4829-450a-81ac-0e8f853ce427
5,How have inference-scaling reasoning models an...,"Yo, the advancements in large language models ...",None,Inference-scaling reasoning models have been a...,1,0,1,4.317529,71a343ef-936b-4ad6-addf-be5bdae84fe4,76e30892-f5a5-4ab5-861a-224773dfd0df
6,How have inference-scaling reasoning models an...,"Yo, so check it out! Inference-scaling reasoni...",None,Inference-scaling reasoning models have been a...,1,0,1,2.976590,541fd9c7-9998-42bc-bd4a-fb4f5e96059a,e03db59a-c709-40ea-856d-3992277d4356
7,How have inference-scaling reasoning models an...,"Yo, check it! In 2024, inference-scaling reaso...",None,"In 2024, the advancements in large language mo...",1,0,1,4.000314,9cecc220-6ddf-45d8-8af9-ef3e106ba88e,ae8114fa-0dea-45f1-805e-ea1f2bd2958c
8,Wht are the Plausible analytics insights from ...,I don't know.,None,The AI Research Analyst used Plausible analyti...,0,0,0,0.799760,ad4d804f-9984-4feb-b4cd-7542f6cb7380,54037fea-01d9-44e5-8a63-fed67dc61226
9,Wht are the key insights from Simon Willison's...,"Yo, check it out! In 2023, Simon Willison drop...",None,Simon Willison's Weblog highlights that 2023 w...,1,1,1,5.369541,b148588d-9c6d-4726-9136-5c26eef82d50,764f8249-7cac-4a6f-949b-f1d25294734f


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

first is the generic rag chain:

![dope](./images/not_dope.png)


next is the dope rag chain:

![dope](./images/dope1.png)

differences between the two chains:
prompt: rag_chain is the generic prompt; dope_rag_chain is the dope prompt with more fluent, cool responses
chunk size: rag_chain is 500; dope_rag_chain is 1000
embedding model: rag_chain is text-embedding-3-small; dope_rag_chain is text-embedding-3-large

dope_rag_chain improves fluency, retrieval accuracy, and coherence through better prompting, embeddings, and chunking. It might have slightly higher latency due to larger embeddings and chunk size, but it's more accurate and fluent.

